# Retention Policy

In [ ]:
import os
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import *

MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"
BUCKET_BRONZE = "bronze"

## 1 - MINIO: APPLING DATA LIFECYCLE POLICY TO BRONZE ZONE

In [ ]:
# Constants for Days and Durations
DAYS_TO_EXPIRE = 3650  # 10 years
ABORT_MULTIPART_UPLOAD_DAYS = 7

# Initialize the MinIO client
client = Minio(
    "minio:9000",  # Replace with your MinIO server endpoint
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False  # Set to True if using HTTPS
)

# Lifecycle policy in XML format
lifecycle_policy = f"""<LifecycleConfiguration>
    <Rule>
        <ID>ExpireObjects</ID>
        <Filter>
            <Prefix></Prefix>
        </Filter>
        <Status>Enabled</Status>
        <Expiration>
            <Days>{DAYS_TO_EXPIRE}</Days>
        </Expiration>
    </Rule>
    <Rule>
        <ID>AbortIncompleteMultipartUpload</ID>
        <Filter>
            <Prefix></Prefix>
        </Filter>
        <Status>Enabled</Status>
        <AbortIncompleteMultipartUpload>
            <DaysAfterInitiation>{ABORT_MULTIPART_UPLOAD_DAYS}</DaysAfterInitiation>
        </AbortIncompleteMultipartUpload>
    </Rule>
</LifecycleConfiguration>"""

# Apply the lifecycle policy to the bucket
client.set_bucket_lifecycle(BUCKET_BRONZE, lifecycle_policy)
print(f"Lifecycle policy applied to bucket '{BUCKET_BRONZE}'.")


## 2 - AWS S3

In [ ]:
import json

# Constants for Days and Durations
DAYS_TO_INFREQUENT_ACCESS = 30
DAYS_TO_GLACIER = 90
DAYS_TO_EXPIRE = 3650  # 10 years
NONCURRENT_VERSION_DAYS = 30
NONCURRENT_VERSION_EXPIRATION_DAYS = 365
ABORT_MULTIPART_UPLOAD_DAYS = 7

# Define the lifecycle policy using constants
lifecycle_policy = {
    "Rules": [
        {
            "ID": "TransitionToInfrequentAccess",
            "Filter": {
                "Prefix": ""
            },
            "Status": "Enabled",
            "Transitions": [
                {
                    "Days": DAYS_TO_INFREQUENT_ACCESS,
                    "StorageClass": "STANDARD_IA"
                }
            ],
            "NoncurrentVersionTransitions": [
                {
                    "NoncurrentDays": NONCURRENT_VERSION_DAYS,
                    "StorageClass": "STANDARD_IA"
                }
            ],
            "NoncurrentVersionExpiration": {
                "NoncurrentDays": NONCURRENT_VERSION_EXPIRATION_DAYS
            },
            "AbortIncompleteMultipartUpload": {
                "DaysAfterInitiation": ABORT_MULTIPART_UPLOAD_DAYS
            }
        },
        {
            "ID": "TransitionToGlacier",
            "Filter": {
                "Prefix": ""
            },
            "Status": "Enabled",
            "Transitions": [
                {
                    "Days": DAYS_TO_GLACIER,
                    "StorageClass": "GLACIER"
                }
            ],
            "NoncurrentVersionTransitions": [
                {
                    "NoncurrentDays": DAYS_TO_GLACIER,
                    "StorageClass": "GLACIER"
                }
            ]
        },
        {
            "ID": "ExpireObjects",
            "Filter": {
                "Prefix": ""
            },
            "Status": "Enabled",
            "Expiration": {
                "Days": DAYS_TO_EXPIRE
            },
            "NoncurrentVersionExpiration": {
                "NoncurrentDays": DAYS_TO_EXPIRE
            }
        }
    ]
}

# Write the lifecycle policy to a JSON file
with open('lifecycle.json', 'w') as f:
    json.dump(lifecycle_policy, f, indent=2)

print("Lifecycle policy JSON file generated.")
lifecycle_policy
